# Study 826 — Treasury Duration BAB 🏦📉

**Does *betting-against-beta* earn a low-risk alpha inside the Treasury curve?**

Frazzini & Pedersen (2014) show that low-beta assets beat high-beta assets on a
*risk-adjusted* basis, so a **BAB** book — long the low-beta legs levered to unit beta,
short the high-beta legs, beta-neutral — earns a positive alpha; they document it across
asset classes, **US Treasuries by maturity included**. We rebuild the Treasury-curve
version from five iShares ETFs that ladder the curve —
SHY (1-3y) → IEI → IEF → TLH → TLT (20y+) — estimate each ETF's beta to an equal-weight
**duration factor**, and form the classic rank-weighted BAB book
(2010-01-04 → 2026-06-30, 5 ETFs).

*Numbers below are the frozen headline (`docs/results.md`); the live cells run the fast
synthetic control. Fingerprint `e423191a2863`.*


## 1. The idea in one picture

Leverage-constrained investors who want more return can't just borrow — so they over-buy high-beta (here: long-duration) assets and bid them up, leaving low-beta (short-duration) assets *cheap per unit of risk*. Frazzini-Pedersen's fix: **lever up the boring low-beta leg** to the same risk as the exciting high-beta leg, short the high-beta leg, and pocket the low-risk premium. Inside the Treasury curve that means: lever up SHY/IEI, short TLH/TLT, beta-neutral.

In [1]:
import numpy as np, pandas as pd
R = dict(bab_bps=1.31, t_nw=2.5, sharpe=0.6, lev_lo_bps=2.0, lev_hi_bps=0.69,
         beta_lo=0.238, beta_hi=1.864, gross_lev=5.04)
print('BAB book: long low-beta (levered) / short high-beta, beta-neutral')
print('  BAB spread : %+.2f bps/day  (Newey-West t = %+.2f, Sharpe %.2f)'
      % (R['bab_bps'], R['t_nw'], R['sharpe']))
print('  levered legs: low-beta %+.2f vs high-beta %+.2f bps/day'
      % (R['lev_lo_bps'], R['lev_hi_bps']))
print('  the cage   : beta_lo %.2f / beta_hi %.2f -> %.1fx gross leverage'
      % (R['beta_lo'], R['beta_hi'], R['gross_lev']))

BAB book: long low-beta (levered) / short high-beta, beta-neutral
  BAB spread : +1.31 bps/day  (Newey-West t = +2.50, Sharpe 0.60)
  levered legs: low-beta +2.00 vs high-beta +0.69 bps/day
  the cage   : beta_lo 0.24 / beta_hi 1.86 -> 5.0x gross leverage


## 2. Is the sort just lucky? A live synthetic control

We plant a Frazzini-Pedersen low-beta alpha in a seeded toy curve (`edge>0`) and check the detector recovers it — and stays *silent* on the null (`edge=0`, betas spread out but no alpha). No network.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from duration_bab import data, strategy as st
null = st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=826, n_days=1300))
planted = st.synthetic_detect(data.synthetic_panel(edge=0.0015, seed=826, n_days=1600))
print('null world   : BAB NW t = %+.2f  (should be ~0)' % null['t_nw'])
print('planted world: BAB NW t = %+.2f  (should light up)' % planted['t_nw'])
print('planted book residual beta to factor = %+.3f (beta-neutral)' % planted['beta_resid'])

null world   : BAB NW t = +0.69  (should be ~0)
planted world: BAB NW t = +30.79  (should light up)
planted book residual beta to factor = +0.006 (beta-neutral)


## 3. The honest verdict — the low-risk alpha does *not* hold here

On the real curve the BAB book prints **+1.31 bps/day** with Newey-West *t* = **+2.50** — right sign, and it *nominally* clears the significance bar. But two checks knock it down:

1. **The placebo refutes the signal.** Permute which ETF's return lands in each leg of the *same* leverage cage: the random assignment earns **more** (placebo mean +2.57 bps) than the real beta-sorted book (+1.31 bps) — the observed sits ~1.7σ into the *left* tail (right-tail p = 0.99). The **beta signal adds no value**; the small positive number is mechanical *levered carry* from the 1/β scaling on the low-vol leg.
2. **It's one era.** 2010–2017 the BAB is flat (*t* = +0.49); the whole result lives in 2018–2026 (*t* = +2.68).

The betas ladder cleanly (SHY 0.12 → TLT 2.09) and the book is beta-neutral (residual β = -0.007), so the machinery is sound — the claimed Frazzini-Pedersen low-risk edge is simply **absent as a signal** on this curve. **Signal: None**, **Tradability: Mirage**.